# mini-infer: final benchmark sweep (week 5)
Every final number from one GPU (Kaggle T4, GPU 0 only, one job at a time):
1. correctness gate (our engine vs Hugging Face, token for token)
2. `bench.final_sweep`: throughput vs users, TTFT vs arrival rate, ablations (3 repeats each, median)
3. week 1 / week 4 benchmarks re-run on the same GPU: KV cache speedup, INT8, speculative decoding

Results and figures are written straight into `/kaggle/working`, so a failure late in the run keeps everything finished before it.

In [ ]:
import os, subprocess
REPO, SRC, OUT = "https://github.com/Vansh8010-boop/LLM-Inference-Engine.git", "/tmp/mini-infer", "/kaggle/working"
if os.path.isdir(SRC):
    subprocess.run(["git", "-C", SRC, "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", REPO, SRC], check=True)
os.chdir(SRC)
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

# Results and figures go straight to the notebook output (survives a later crash).
for repo_dir, out_dir in (("bench/results", f"{OUT}/results"), ("docs/figures", f"{OUT}/figures")):
    os.makedirs(out_dir, exist_ok=True)
    subprocess.run(["rm", "-rf", repo_dir], check=True)
    os.symlink(out_dir, repo_dir)
os.makedirs("models", exist_ok=True)

In [ ]:
# Same transformers as the local runs; the rest is the serving + benchmark stack.
!pip install -q "transformers==5.20.2" fastapi "uvicorn[standard]" httpx sse-starlette datasets 2>&1 | tail -3
!python -c "import torch, transformers; print('torch', torch.__version__, '| transformers', transformers.__version__, '| gpus', torch.cuda.device_count(), torch.cuda.get_device_name(0))"
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader
!nproc; free -g | head -2

In [ ]:
# Weights: Kaggle's network makes this a couple of minutes.
os.environ.update({"HF_HUB_DISABLE_PROGRESS_BARS": "1", "TQDM_DISABLE": "1", "PYTHONUNBUFFERED": "1",
                   "CUDA_VISIBLE_DEVICES": "0"})
for repo in ("Qwen/Qwen2.5-0.5B-Instruct", "Qwen/Qwen2.5-1.5B-Instruct"):
    subprocess.run(["python", "scripts/download_model.py", "--model", repo, "--no-test"], check=True)
!ls -la models/*

In [ ]:
# Correctness gate on this GPU before measuring anything: exact HF match + batching + scheduler.
!MINI_INFER_TEST_PROMPTS=20 python -m pytest -q -p no:cacheprovider tests/test_block_manager.py tests/test_scheduler.py tests/test_correctness.py tests/test_batched.py 2>&1 | tail -5

In [ ]:
# The sweep: ~3-4 h. 16 GB card, so a larger GPU cap and KV pool than the laptop run.
p = subprocess.run(["python", "-m", "bench.final_sweep", "--gpu-fraction", "0.85", "--kv-cache-gb", "4",
                    "--cooldown", "20"], stdout=open(f"{OUT}/sweep_stdout.log", "w"), stderr=subprocess.STDOUT)
print("sweep exit code", p.returncode)
print(open(f"{OUT}/results/sweep_logs/sweep.log").read())

In [ ]:
# Week 1 and week 4 benchmarks again, so every final number shares one GPU.
for mod in ("bench.kv_cache_speedup", "bench.quant_bench", "bench.speculative_bench"):
    p = subprocess.run(["python", "-m", mod], stdout=open(f"{OUT}/{mod.split('.')[1]}.log", "w"),
                       stderr=subprocess.STDOUT)
    print(mod, "exit code", p.returncode)
    print("".join(l for l in open(f"{OUT}/{mod.split('.')[1]}.log") if l.startswith("[") or "Error" in l)[-2000:])

In [ ]:
!ls -la {OUT}/results {OUT}/figures